# Morteza Project: IMU and xdf file



We have a file in XDF format containing kinematic IMU data. This format, due to its sampling method, helps ensure that the recording time for each sensor is identical and prevents timing discrepancies by synchronizing the recording timestamps.

Objective: To read and analyze the file.


In [1]:
# Basic Packages
import os

# xdf reader
import pyxdf # pip install pyxdf

# Analysis Packages
import numpy as np
import pandas as pd

# Plot Packages
import matplotlib.pyplot as plt
import seaborn as sns

# Static Package
import scipy.stats as stats

### PATH of input and output

In [2]:
PATH = os.path.dirname(os.getcwd())
NAME_FOLDER_INPUT = r"data\raw\6. Mrteza Project IMU and xdf file\TEST 27 MAI"
NAME_FOLDER_OUTPUT = r"data\processed\6. Mrteza Project IMU and xdf file\TEST 27 MAI"

# Make input and output path
INPUT_PATH = os.path.join(PATH, NAME_FOLDER_INPUT)
INPUT_OUTPUT = os.path.join(PATH, NAME_FOLDER_OUTPUT)

### Read name of all file by `xdf` format

create one dataframe from name of all data   
create one column from name of dataframe   
`inf_all_sdf_data` is a dataframe from file name and name of each dataframe

In [3]:
# read name of each file as .xdf
name_all_file = [f for f in os.listdir(INPUT_PATH) if f.endswith(".xdf")]

# create datafram from name of data by .xdf
inf_all_sdf_data = pd.DataFrame(name_all_file, columns=["name_file"])

# create name of each dataframe
inf_all_sdf_data["name_df"] = (inf_all_sdf_data["name_file"].
                               str.replace(".xdf", "").
                               str.replace("-","_").
                               str.replace(" ", "_"))
inf_all_sdf_data

,name_file,name_df
0,abduction.xdf,abduction
1,flexionandextension.xdf,flexionandextension
2,joggen.xdf,joggen
3,squat.xdf,squat
4,standing.xdf,standing


### 1. Read all data

In [7]:
# fuction reader file
def path_each_file(name_each_file:str, name_each_df:str):
    """function **path_each_file**   
    give 2 parametr   
    name_each_file: .xdf   
    name_each_df: name of each dataframe
    """
    path_read = os.path.join(INPUT_PATH, name_each_file)
    globals()[name_each_df], header = pyxdf.load_xdf(path_read)
    return globals()[name_each_df]



# maping file for sensor name
maping_file = [f for f in os.listdir(INPUT_PATH) if f.endswith(".xlsx")]
PATH_MAP_SENSOR = os.path.join(INPUT_PATH, maping_file[0])
MAP_SENSOR = pd.read_excel(PATH_MAP_SENSOR, sheet_name="Tabelle1")

# create all dataframe
name_dataframe = []
for each_df in range(inf_all_sdf_data.shape[0]):
    name_each_file = inf_all_sdf_data.iloc[each_df]["name_file"]
    name_each_df = inf_all_sdf_data.iloc[each_df]["name_df"]
    path_each_file(name_each_file, name_each_df)
    
    data = globals()[name_each_df]
    valid_streams = [s for s in data if len(s['time_series']) > 0] # mask data, time_series is not empty
    
    for stream in valid_streams:
        
        # name sensor by mapping file
        name_df= stream['info']['name'][0]
        name_sensor = MAP_SENSOR[MAP_SENSOR["ID"].eq(name_df[-8:])]["Name"].iloc[0]
        #---------------------
        name_dataframe.append(f"{name_each_df}_{name_sensor}") # name of each df
        
        df = pd.DataFrame(stream['time_series']) # create df by time_series
        df.index = stream['time_stamps'] # set index by time_stamps
        df.index.name = 'timestamp' # set name of index
        df.columns = [ch['label'][0] for ch in stream['info']['desc'][0]['channels'][0]['channel']] # set name of each column
        globals()[f"{name_each_df}_{name_sensor}"] = df # create df by orginal name





Stream 1: last clock offset is statistically anomalous, truncating (see pylsl#67, liblsl#246).
Stream 1: sample count (1976) exceeds footer sample_count (1975), truncating extra samples.


### Create info IMU df

In [8]:
inf_IMU = pd.DataFrame(name_dataframe, columns=["name_dataframe"])
inf_IMU["name_sensor"] = inf_IMU["name_dataframe"].str.split("_", n=1).str[0]
inf_IMU["name_loc_sensor"] = inf_IMU["name_dataframe"].str.split("_", n=1).str[1]

In [9]:
inf_IMU["name_loc_sensor"].unique()

array(['pelvis', 'femur_l', 'tibia_l', 'calcn_l'], dtype=object)

In [ ]:
inf_IMU[inf_IMU["name_loc_sensor"].eq("pelvis")]


,name_dataframe,name_sensor,name_loc_sensor
0,abduction_pelvis,abduction,pelvis
4,flexionandextension_pelvis,flexionandextension,pelvis
9,joggen_pelvis,joggen,pelvis
12,squat_pelvis,squat,pelvis
16,standing_pelvis,standing,pelvis


In [14]:
joggen_tibia_l

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,mx,my,mz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,,,,
517311.976676,-0.636793,0.334667,-0.635899,-0.279510,-9.791249,-0.674528,-0.440735,-0.018265,-0.015218,0.018459,0.716797,0.267578,0.340576,10419.0,0.0
517311.986729,-0.636784,0.334681,-0.635798,-0.279744,-9.767352,-0.702161,-0.477939,-0.035649,-0.025765,0.019072,0.719727,0.258301,0.352295,10420.0,0.0
517311.996782,-0.636766,0.334695,-0.635707,-0.279974,-9.727255,-0.703493,-0.463575,-0.034773,-0.023761,0.019254,0.715820,0.260742,0.344482,10421.0,0.0
517312.006834,-0.636765,0.334672,-0.635634,-0.280166,-9.743767,-0.727655,-0.512767,-0.023485,-0.022564,0.019690,0.711182,0.261963,0.343750,10422.0,0.0
517312.016887,-0.636803,0.334620,-0.635542,-0.280354,-9.734570,-0.704395,-0.535764,-0.018013,-0.031991,0.019277,0.718506,0.267578,0.347900,10423.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517328.975720,-0.541553,0.453713,-0.607865,-0.362443,-9.663155,-0.470978,-1.421263,0.039437,-0.028084,-0.019001,0.629395,-0.596436,0.283447,12119.0,0.0
517328.985773,-0.541668,0.453639,-0.607857,-0.362378,-9.683144,-0.429190,-1.392736,0.028564,-0.019776,-0.011255,0.635742,-0.586426,0.291260,12120.0,0.0
517328.995825,-0.541757,0.453631,-0.607777,-0.362388,-9.706743,-0.407599,-1.401393,0.004208,-0.020101,-0.014474,0.626465,-0.598389,0.283691,12121.0,0.0


### Read one data

In [4]:
# path of one data
path_read = os.path.join(INPUT_PATH, inf_all_sdf_data["name_file"].iloc[0])

# raed data
data, header = pyxdf.load_xdf(path_read)

In [5]:
header

{'info': defaultdict(list,
             {'version': ['1.0'], 'datetime': ['2026-05-27T17:17:06+0200']})}

In [11]:
name_each_Xsens = []
for i in range(len(data)):
    name_each_Xsens.append(data[i]["info"]["name"])

name_each_Xsens

[['Xsens_MTw2_00B4D0C5'],
 ['Xsens_MTw2_00B4D0C8'],
 ['Xsens_MTw2_00B4D0BE'],
 ['Xsens_MTw2_00B4D0D0']]

In [13]:
data[0]['info']['desc'][0]['channels'][0]['channel']

[defaultdict(list, {'label': ['qw']}),
 defaultdict(list, {'label': ['qx']}),
 defaultdict(list, {'label': ['qy']}),
 defaultdict(list, {'label': ['qz']}),
 defaultdict(list, {'label': ['ax']}),
 defaultdict(list, {'label': ['ay']}),
 defaultdict(list, {'label': ['az']}),
 defaultdict(list, {'label': ['gx']}),
 defaultdict(list, {'label': ['gy']}),
 defaultdict(list, {'label': ['gz']}),
 defaultdict(list, {'label': ['mx']}),
 defaultdict(list, {'label': ['my']}),
 defaultdict(list, {'label': ['mz']}),
 defaultdict(list, {'label': ['packet_counter']}),
 defaultdict(list, {'label': ['sample_time_fine']})]

In [125]:
data[0].keys(), data[6].keys()

(dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values']),
 dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values']))

In [ ]:
stream = data[0]
df = pd.DataFrame(stream['time_series'])
df.index = stream['time_stamps']
df.index.name = 'timestamp'
df.columns = [ch['label'][0] for ch in stream['info']['desc'][0]['channels'][0]['channel']]
df

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,
229501.472544,0.099020,0.005810,-0.002649,-0.995065,-0.090631,0.075291,9.776200,-0.008962,0.006068,-0.002955,12303.0,0.0
229501.482596,0.099021,0.005805,-0.002610,-0.995065,-0.083169,0.051323,9.751882,-0.006871,0.003363,-0.001430,12304.0,0.0
229501.492647,0.099018,0.005803,-0.002555,-0.995066,-0.091469,0.073681,9.776204,-0.010151,0.004301,-0.001368,12305.0,0.0
229501.502698,0.099004,0.005797,-0.002484,-0.995067,-0.101624,0.094862,9.751747,-0.013208,0.003712,-0.004408,12306.0,0.0
229501.512749,0.098985,0.005792,-0.002421,-0.995069,-0.115895,0.062052,9.752830,-0.011709,0.003993,-0.004440,12307.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
229515.453850,0.083752,0.011817,-0.001658,-0.996415,-0.184272,0.071303,9.766708,0.003579,0.005374,-0.003182,13702.0,0.0
229515.463901,0.083762,0.011826,-0.001646,-0.996414,-0.172329,0.103592,9.766027,-0.001278,0.005996,0.001436,13703.0,0.0
229515.473952,0.083761,0.011856,-0.001644,-0.996414,-0.175751,0.049081,9.767643,0.001122,0.009563,-0.001724,13704.0,0.0


In [151]:
for i in [0, 6]:
    s = data[i]
    print(f"--- Stream {i} ---")
    print(f"Name: {s['info']['name']}")
    print(f"Type: {s['info']['type']}")
    print(f"Channel Count: {s['info']['channel_count']}")
    print(f"Series shape: {len(s['time_series'])}")
    print(f"Timestamps shape: {len(s['time_stamps'])}")
    print("-" * 20)

--- Stream 0 ---
Name: ['Xsens_MTw2_00B4D0C8']
Type: ['IMU']
Channel Count: ['12']
Series shape: 1396
Timestamps shape: 1396
--------------------
--- Stream 6 ---
Name: ['Xsens_MTw2_00B4D0C8']
Type: ['IMU']
Channel Count: ['12']
Series shape: 0
Timestamps shape: 0
--------------------


In [169]:
valid_streams = [s for s in data if len(s['time_series']) > 0]
name_dataframe = []
for stream in valid_streams:
    print(f"name dataframe: {stream['info']['name']}")
    name_dataframe.append(stream['info']['name'])
    globals()[stream['info']['name'][0]] = pd.DataFrame(stream['time_series'])
    globals()[stream['info']['name'][0]].index = stream['time_stamps']
    globals()[stream['info']['name'][0]].index.name = 'timestamp'
    globals()[stream['info']['name'][0]].columns = [ch['label'][0] for ch in stream['info']['desc'][0]['channels'][0]['channel']]
   

name dataframe: ['Xsens_MTw2_00B4D0C8']
name dataframe: ['Xsens_MTw2_00B4D0BE']
name dataframe: ['Xsens_MTw2_00B4D0C5']
name dataframe: ['Xsens_MTw2_00B4D0C4']
name dataframe: ['Xsens_MTw2_00B4D0D0']
name dataframe: ['Xsens_MTw2_00B4D0BF']


In [170]:
name_dataframe

[['Xsens_MTw2_00B4D0C8'],
 ['Xsens_MTw2_00B4D0BE'],
 ['Xsens_MTw2_00B4D0C5'],
 ['Xsens_MTw2_00B4D0C4'],
 ['Xsens_MTw2_00B4D0D0'],
 ['Xsens_MTw2_00B4D0BF']]

In [178]:
Xsens_MTw2_00B4D0C8.head(2)

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,
229501.472544,0.099020,0.005810,-0.002649,-0.995065,-0.090631,0.075291,9.776200,-0.008962,0.006068,-0.002955,12303.0,0.0
229501.482596,0.099021,0.005805,-0.002610,-0.995065,-0.083169,0.051323,9.751882,-0.006871,0.003363,-0.001430,12304.0,0.0


In [ ]:
Xsens_MTw2_00B4D0BE.head(2)

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,
229501.461992,0.334465,0.003907,-0.000751,-0.942400,-0.064216,0.030809,9.730543,0.001064,0.005699,-0.002444,12303.0,0.0
229501.472052,0.334455,0.003904,-0.000747,-0.942403,-0.056171,0.045107,9.700230,0.000548,0.002237,-0.005045,12304.0,0.0


### Reading all data streams simultaneously.   
Using the `inf_all_sdf_data` DataFrame, we read the individual dataframes and create a specific dataset for each, named accordingly.


In [43]:
for file in inf_all_sdf_data["name_file"]:
    print(file)

sub-P001_ses-S001_task-Default_run-001_eeg.xdf
